# RAG Query: Before and After

Ask an LLM a question **without RAG** (vague / wrong), then **with RAG**
(accurate, cited). Run `rag_ingestion.ipynb` first to populate Milvus.

### Prerequisites

This notebook requires two services beyond Milvus:

1. **vLLM inference endpoint** — a separately deployed vLLM model serving
   instance (e.g. Mistral-7B-Instruct) exposed as an OpenAI-compatible API.
   Deploy one via the RHOAI Model Serving UI or a vLLM `InferenceService` CR.
   Set `INFERENCE_URL` and `MODEL_NAME` in the Configure cell below.

2. **Embedding model** — `sentence-transformers` loads the embedding model
   locally in this notebook to encode your query. The model **must match**
   the one used during ingestion (`VLLM_MODEL_SOURCE` in `rag_ingestion.ipynb`).
   The default is `intfloat/multilingual-e5-large`.

In [ ]:
%pip install -q pymilvus sentence-transformers openai torch

## Configure

In [ ]:
# Milvus connection (must match ingestion settings)
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
MILVUS_DB = "default"
COLLECTION_NAME = "rag_documents"

# Must match the model used during ingestion (VLLM_MODEL_SOURCE in rag_ingestion.ipynb).
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

# Set these to your vLLM inference endpoint and model name.
# Deploy a model via the RHOAI Model Serving UI or a vLLM InferenceService CR.
INFERENCE_URL = "http://<your-vllm-service>.<namespace>.svc.cluster.local:8080/v1"
MODEL_NAME = "<your-model-name>"
TOP_K = 5

## Initialize clients

In [ ]:
from openai import OpenAI
from pymilvus import MilvusClient
from rag_helpers import ask_llm, build_context, print_comparison, search_milvus
from sentence_transformers import SentenceTransformer

milvus = MilvusClient(uri=f"http://{MILVUS_HOST}:{MILVUS_PORT}", db_name=MILVUS_DB)
embed_model = SentenceTransformer(EMBEDDING_MODEL)
llm = OpenAI(base_url=INFERENCE_URL, api_key="unused")

milvus.load_collection(COLLECTION_NAME)
stats = milvus.get_collection_stats(COLLECTION_NAME)
row_count = stats.get("row_count", stats.get("data", {}).get("row_count", "?"))
print(f"Collection '{COLLECTION_NAME}': {row_count} vectors")

## Pick a question

In [ ]:
# Change this to a question relevant to the PDFs you ingested.
# If you used fetch_sample_pdfs.sh, the papers cover topics like
# language models, retrieval-augmented generation, and embeddings.
QUESTION = "How does retrieval-augmented generation improve the accuracy of language model responses?"
print(QUESTION)

## Query WITHOUT RAG

In [ ]:
answer_no_rag = ask_llm(QUESTION, llm=llm, model_name=MODEL_NAME)
print(answer_no_rag)

## Retrieve context from Milvus

In [ ]:
chunks = search_milvus(
    QUESTION,
    milvus=milvus,
    embed_model=embed_model,
    collection_name=COLLECTION_NAME,
    top_k=TOP_K,
)

for i, c in enumerate(chunks, 1):
    preview = c["text"][:100].replace("\n", " ")
    print(f"[{i}] {c['source_file']}  chunk {c['chunk_index']}  score {c['score']:.3f}")
    print(f"    {preview}...\n")

## Query WITH RAG

In [ ]:
context = build_context(chunks)
answer_with_rag = ask_llm(QUESTION, llm=llm, model_name=MODEL_NAME, context=context)
print(answer_with_rag)

## Compare side by side

In [ ]:
print_comparison(QUESTION, answer_no_rag, answer_with_rag, chunks)